# Reading a plan, and checking a job

`explain(analyze=True)` runs the query and prints each operator's rows, time, peak memory and spill bytes. `krishiv check-compat` scans PySpark and SQL files for functions Krishiv does not run yet.

In [ ]:
import pathlib, sys, tempfile
sys.path.insert(0, "../datasets")  # docs/notebooks -> docs/datasets
import generate
import krishiv
from krishiv.sql import functions as F

data = pathlib.Path(tempfile.mkdtemp())
generate.generate(data, orders=20_000, customers=500)
spark = krishiv.session()
orders = spark.read.parquet(str(data / "orders.parquet"))
customers = spark.read.parquet(str(data / "customers.parquet"))
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

In [ ]:
q = orders.groupBy("customer_id").agg(F.sum("amount").alias("spent"))
q.explain()  # the physical plan

In [ ]:
q.explain(analyze=True)

In [ ]:
plan = spark.sql("EXPLAIN ANALYZE SELECT customer_id, sum(amount) FROM orders GROUP BY customer_id").collect()[0][0]
assert plan.startswith("== Physical Plan (analyzed) ==")

From a shell, `krishiv check-compat .` lists the unsupported functions a project calls; see the [migration guide](../guides/pyspark-migration.md).